In [1]:
import json
import base64
from openai import OpenAI

In [2]:
client = OpenAI(
    api_key="os.environ.get("OPENAI_API_KEY")"
)

In [3]:
import json

with open('/home/as5606/Datasets/cholec_formatted_data/cholec80_llava_test.json', 'r') as f:
    original_data = json.load(f)

In [4]:
import json

with open('/home/as5606/projects/Hulu-Med/gemini/gemini_test_answers.json', 'r') as f:
    gemini_data = json.load(f)


In [5]:
SYSTEM_PROMPT = """\
You are an answer-matching evaluator for surgical VQA.

Your task is to compare two answers and decide whether they have the same meaning.

The answers do not need to use the exact same words. They should be considered a match if they express the same core idea, refer to the same entity, or would be accepted as semantically equivalent in context.

You should focus on meaning, not wording.

Return only one of the following labels:

MATCH
NO_MATCH

Guidelines:
- Mark MATCH if both answers convey the same meaning, even with different wording.
- Mark MATCH if one answer is more detailed but does not contradict the other.
- Mark MATCH for synonyms, paraphrases, abbreviations, or equivalent medical/scientific terms.
- Mark MATCH for number variations: "2" matches "Two" or "There are 2 tools"
- Mark MATCH when one answer elaborates on the other without contradiction
- Mark MATCH for yes/no variations: "No" matches "X is not used" or "X was not present"
- Mark MATCH for yes/no variations: "Yes" matches "X is used" or "X is present"
- Mark MATCH for negation patterns: "No" matches "There are no scissors visible"
- Mark NO_MATCH if the answers refer to different entities, objects, actions, locations, numbers, or concepts.
- Mark NO_MATCH if one answer contradicts the other.
- Mark NO_MATCH if one answer is too vague to confirm equivalence.
- Ignore capitalization, punctuation, grammar mistakes, and minor wording differences.

CRITICAL EXAMPLES:
Answer 1: "2" | Answer 2: "There are 2 tools operating in this image." → MATCH (same number)
Answer 1: "specimen bag is not used in clipping cutting" | Answer 2: "No." → MATCH (both say no/not used)
Answer 1: "Yes" | Answer 2: "The grasper is used in preparation." → MATCH (both say yes/is used)
Answer 1: "1" | Answer 2: "One tool" → MATCH (number equivalence)
Answer 1: "Left" | Answer 2: "The tool is on the left side" → MATCH (same location)

Input format:
Answer 1: <first answer>
Answer 2: <second answer>

Output format:
MATCH or NO_MATCH
"""


In [6]:
def _parse_json(raw: str) -> dict:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

In [7]:
def _encode_image(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

In [8]:
def compare_answers(answer1: str, answer2: str) -> str:
    """Compare two answers using ChatGPT"""
    user_text = f"Answer 1: {answer1}\nAnswer 2: {answer2}"
    
    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=SYSTEM_PROMPT,
            input=user_text,
        )
        
        result = response.output_text.strip()
        
        # Extract MATCH or NO_MATCH
        if "MATCH" in result.upper():
            if "NO_MATCH" in result.upper():
                no_match_idx = result.upper().find("NO_MATCH")
                match_idx = result.upper().find("MATCH")
                if no_match_idx < match_idx:
                    return "NO_MATCH"
            return "MATCH"
        else:
            return "NO_MATCH"
    
    except Exception as e:
        print(f"Error comparing answers: {e}")
        return None

In [9]:
# Create mapping of ID to gemini answers for quick lookup
gemini_map = {item['id']: item for item in gemini_data}

In [10]:
from tqdm import tqdm

print("Comparing answers with ChatGPT...\n")
results = []

for original_item in tqdm(original_data, desc="Processing"):
    item_id = original_item['id']
    
    # Get original answer
    original_answer = original_item['conversations'][1]['value']
    question = original_item['conversations'][0]['value'].replace('\n<image>', '').strip()
    
    # Get gemini answer
    if item_id in gemini_map:
        gemini_answer = gemini_map[item_id]['gemini_answer']
        
        # Compare
        match = compare_answers(original_answer, gemini_answer)
        
        # Store result
        result = {
            "id": item_id,
            "image": original_item['image'],
            "question": question,
            "original_answer": original_answer,
            "gemini_answer": gemini_answer,
            "match": match
        }
        results.append(result)

# Save results
output_path = '/home/as5606/projects/Hulu-Med/gemini/gemini_test_accuracy_comparison.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)


Comparing answers with ChatGPT...



Processing: 100%|██████████| 7652/7652 [1:32:34<00:00,  1.38it/s]  
